In [2]:
%%capture
!pip install transformers[torch]
!pip install rouge_score
!pip install datasets
!pip install evaluate

In [17]:
from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer
)
import evaluate
import numpy as np

In [18]:
from datasets import load_dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [6]:
dataset['train'].features

{'article': Value('string'),
 'highlights': Value('string'),
 'id': Value('string')}

In [7]:
# Taking the subset of the dataset for the finetuning purpose
train_subset = dataset['train'].select(range(10000))
validation_subset = dataset['validation'].select(range(1000))
test_subset = dataset["test"].select(range(1000))

In [8]:
checkpoint = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(checkpoint)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [9]:
def preprocess_function(examples):
    inputs = ["summarize:" + doc for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True)

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["highlights"] , max_length=150, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [10]:
# Tokenize datasets
tokenized_train = train_subset.map(preprocess_function, batched=True)
tokenized_validation = validation_subset.map(preprocess_function, batched=True)
tokenized_test = test_subset.map(preprocess_function, batched=True)

In [11]:
# Load Model
model = T5ForConditionalGeneration.from_pretrained(checkpoint)

In [12]:
# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [13]:
import numpy as np
rouge = evaluate.load("rouge")

def compute_metrics(pred):

    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_ids = np.where(
        pred_ids != -100,
        pred_ids,
        tokenizer.pad_token_id
    )

    labels_ids = np.where(
        labels_ids != -100,
        labels_ids,
        tokenizer.pad_token_id
    )

    pred_str = tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    label_str = tokenizer.batch_decode(
        labels_ids,
        skip_special_tokens=True
    )

    rouge_output = rouge.compute(
        predictions=pred_str,
        references=label_str,
        use_stemmer=True
    )

    result = {
        key: value * 100
        for key, value in rouge_output.items()
    }

    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id)
        for pred in pred_ids
    ]

    result["gen_len"] = np.mean(prediction_lens)

    return result

In [14]:
# Seq2Seq training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=150,
    generation_num_beams=4,
    fp16=True
)

In [15]:
## Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,                       # The model to be trained
    args=training_args,                # Training arguments defined with Seq2SeqTrainingArguments
    train_dataset=tokenized_train,     # The training dataset
    eval_dataset=tokenized_validation, # The evaluation dataset
    data_collator=data_collator,       # The data collator for processing data batches
    tokenizer=tokenizer,               # The tokenizer used for preprocessing
    compute_metrics=compute_metrics,   # The function to compute evaluation metrics
)


C:\Users\mishr\AppData\Local\Temp\ipykernel_20936\493624131.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [15]:
# Train the model
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss


NameError: name 'rouge' is not defined

In [17]:
trainer.train(resume_from_checkpoint="./results/checkpoint-1875")

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
C:\Users\mishr\anaconda3\envs\myenv\Lib\site-packages\transformers\trainer.py:3354: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the load

Epoch,Training Loss,Validation Loss


TrainOutput(global_step=1875, training_loss=0.0, metrics={'train_runtime': 0.1252, 'train_samples_per_second': 239592.825, 'train_steps_per_second': 14974.552, 'total_flos': 4060254044160000.0, 'train_loss': 0.0, 'epoch': 3.0})

In [18]:
# Evaluate the model on validation set
trainer.evaluate()

# Evaluate the model on test set
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print(test_results)

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


{'eval_loss': 2.1443588733673096, 'eval_rouge1': 30.968306362232035, 'eval_rouge2': 11.954656949391552, 'eval_rougeL': 22.159633053477673, 'eval_rougeLsum': 22.166377193974085, 'eval_gen_len': 75.412, 'eval_runtime': 328.9913, 'eval_samples_per_second': 3.04, 'eval_steps_per_second': 0.191, 'epoch': 3.0}


In [19]:
import torch
# Select a specific data point from the test dataset
test_index = 78  # Change this index to the specific data point you want to summarize
example_text = dataset["test"][test_index]["article"]

# Preprocess the input text
input_text = "summarize: " + example_text
inputs = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = inputs.to(device)
# Generate the summary
summary_ids = model.generate(inputs, max_length=150, min_length=40, length_penalty=2.0, num_beams=4, early_stopping=True)

# Decode the generated summary
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Original Text:\n", example_text)
print("\nGenerated Summary:\n", summary)

Original Text:
 (CNN)The nation's top stories will be unfolding Tuesday in courthouses and political arenas across the country. Massachusetts is hosting two of the highest-profile court trials in recent memory -- those of former New England Patriot Aaron Hernandez and Boston bombing suspect Dzhokhar Tsarnaev. Both lengthy trials are coming to a close. In Louisville, Kentucky, Sen. Rand Paul made the not-so-surprising announcement that he will run for president, while in Chicago, voters will head to the polls in a very surprising runoff between Mayor Rahm Emanuel and challenger Jesus "Chuy" Garcia. And in Ferguson, Missouri, the shadow of Michael Brown and the protests over his shooting by Officer Darren Wilson will loom large over the city's elections. Here's a breakdown of what to expect today and how we got here: . Tsarnaev, who's accused of detonating a bomb at the 2013 Boston Marathon along with his now-deceased brother, faces the stiffest of penalties -- life in prison or the deat

In [15]:
model.save_pretrained("./my_t5_summarizer")
tokenizer.save_pretrained("./my_t5_summarizer")

('./my_t5_summarizer\\tokenizer_config.json',
 './my_t5_summarizer\\special_tokens_map.json',
 './my_t5_summarizer\\spiece.model',
 './my_t5_summarizer\\added_tokens.json')

In [9]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained(
    "./my_t5_summarizer"
)

tokenizer = T5Tokenizer.from_pretrained(
    "./my_t5_summarizer"
)

In [10]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print(next(model.parameters()).device)

cuda:0


In [20]:
import torch

example_text = """India secured a memorable victory over Australia in the final of a major international cricket tournament. Chasing a target of 289 runs, India lost an early wicket but quickly recovered through a strong partnership between Virat Kohli and Shubman Gill.

Kohli played a patient innings, rotating the strike effectively and punishing loose deliveries. Gill complemented him with aggressive stroke play, keeping the required run rate under control. The pair added more than 140 runs for the second wicket before Gill was dismissed.

Australia attempted a comeback through their fast bowlers, who picked up wickets in the middle overs. However, KL Rahul and Hardik Pandya stabilized the innings and ensured that India remained ahead of the required rate.

The victory sparked celebrations across the country, with fans praising the team's consistency throughout the tournament. Captain Rohit Sharma credited the bowling unit for restricting Australia to a manageable total and highlighted the team's ability to perform under pressure.

Cricket analysts described the match as one of India's most complete performances in recent years. The win strengthened India's position as one of the leading teams in world cricket and boosted confidence ahead of upcoming international competitions.
"""

input_text = "summarize: " + example_text

inputs = tokenizer.encode(
    input_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = inputs.to(device)

summary_ids = model.generate(
    inputs,
    max_length=150,
    min_length=40,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Original Text:\n", example_text)
print("\nGenerated Summary:\n", summary)

Original Text:
 India secured a memorable victory over Australia in the final of a major international cricket tournament. Chasing a target of 289 runs, India lost an early wicket but quickly recovered through a strong partnership between Virat Kohli and Shubman Gill.

Kohli played a patient innings, rotating the strike effectively and punishing loose deliveries. Gill complemented him with aggressive stroke play, keeping the required run rate under control. The pair added more than 140 runs for the second wicket before Gill was dismissed.

Australia attempted a comeback through their fast bowlers, who picked up wickets in the middle overs. However, KL Rahul and Hardik Pandya stabilized the innings and ensured that India remained ahead of the required rate.

The victory sparked celebrations across the country, with fans praising the team's consistency throughout the tournament. Captain Rohit Sharma credited the bowling unit for restricting Australia to a manageable total and highlighted